Data Set Combinging

Imports

In [ ]:
import os
import random
import shutil
from PIL import Image

Genaral Paths & Directory setup

In [ ]:
# Target Paths
datasetDir_path = "../Dataset/Processing/"
resourseDir_path = "../Dataset/Resources/ProcessingDataset/"

target_genuine = os.path.join(datasetDir_path, "HybridDataset/Genuine")
target_forged = os.path.join(datasetDir_path, "HybridDataset/Forged")

# Create target folders if not exist
os.makedirs(os.path.join(datasetDir_path, "HybridDataset"), exist_ok=True)
os.makedirs(target_genuine, exist_ok=True)
os.makedirs(target_forged, exist_ok=True)

print("Completed Dataset Directory Setup")

In [ ]:
# =============================================
# CONVERT TO TESTING DATASET PURPOSE
# =============================================


target_genuine = os.path.join(datasetDir_path, "AdvanceTestingDataset/TestingDataset/Genuine")
target_forged = os.path.join(datasetDir_path, "AdvanceTestingDataset/TestingDataset/Forged")

# Create target folders if not exist
os.makedirs(os.path.join(datasetDir_path, "AdvanceTestingDataset"), exist_ok=True)
os.makedirs(os.path.join(datasetDir_path, "AdvanceTestingDataset/TestingDataset"), exist_ok=True)
os.makedirs(target_genuine, exist_ok=True)
os.makedirs(target_forged, exist_ok=True)


Dataset 00: Archive

In [ ]:
d0_source_test_path = os.path.join(resourseDir_path, "archive/Test")
d0_source_train_path = os.path.join(resourseDir_path, "archive/Train")

def convert_to_jpg(source_path, target_path):
    """Convert an image to .jpg if it's not already"""
    img = Image.open(source_path)
    rgb_img = img.convert("RGB")  # Ensure it's in RGB mode for .jpg
    rgb_img.save(target_path, "JPEG")

def d0_organise(mainPath, file_cnt):
    for dirName in os.listdir(mainPath):
        dirName_path = os.path.join(mainPath, dirName)

        # Check if it's a directory
        if not os.path.isdir(dirName_path):
            print("Error with unidentified resource -", dirName_path)  
            continue  

        for fileName in os.listdir(dirName_path):
            source_file_path = os.path.join(dirName_path, fileName)
            file_base, file_ext = os.path.splitext(fileName)  # Get base name and extension
            file_ext = file_ext.lower()  # Normalize extension case
            new_name = ""

            if "_" in fileName:
                fileNameParts = file_base.split("_")  # Use file_base to avoid extensions
                if len(fileNameParts) > 2 and fileNameParts[0].lower().startswith("ori"):
                    new_name = f"T_D0_{dirName}_{fileNameParts[2]}"

                elif len(fileNameParts) > 2 and fileNameParts[0].lower().startswith("for"):
                    new_name = f"F_D0_{dirName}_{fileNameParts[2]}"

            elif "-" in fileName:
                fileNameParts = file_base.split("-")
                if len(fileNameParts) >= 5:
                    if fileNameParts[3].lower().startswith("g"):
                        new_name = f"T_D0_{fileNameParts[2]}_{fileNameParts[4]}"

                    elif fileNameParts[3].lower().startswith("f"):
                        new_name = f"F_D0_{fileNameParts[2]}_{fileNameParts[4]}"

            if new_name:
                new_name += ".jpg"  # Ensure final filename ends with .jpg
                target_dir = target_genuine if new_name.startswith("T_") else target_forged
                target_file_path = os.path.join(target_dir, new_name)

                if file_ext != ".jpg":
                    convert_to_jpg(source_file_path, target_file_path)
                else:
                    shutil.copy(source_file_path, target_file_path)

                file_cnt += 1

    return file_cnt

d0_file_cnt = 0
d0_file_cnt = d0_organise(d0_source_test_path, d0_file_cnt)
d0_file_cnt = d0_organise(d0_source_train_path, d0_file_cnt)

print("Completed Dataset 2 Process")
print(f"Images Count: {d0_file_cnt}")


Dataset 02: Archive 2

In [ ]:
d2_source_forgery = os.path.join(resourseDir_path, "archive2/FORGERY_ALL")
d2_source_real = os.path.join(resourseDir_path, "archive2/REAL_ALL")

# Processing Forged Signatures
for file_name in os.listdir(d2_source_forgery):
    if file_name.endswith(".jpg"):
        parts = file_name.split("_")
        if len(parts) == 2:
            person_id = parts[0][-3:]  # Extract last 3 digits of the first part as person ID
            image_id = parts[1].split(".")[0]  # Extract image ID before file extension
            new_name = f"F_D2_{person_id}_{image_id}.jpg"
            shutil.copy(os.path.join(d2_source_forgery, file_name), os.path.join(target_forged, new_name))

# Processing Genuine Signatures
for file_name in os.listdir(d2_source_real):
    if file_name.endswith(".jpg"):
        parts = file_name.split("_")
        if len(parts) == 2:
            person_id = parts[0][-3:]  # Extract last 3 digits of the first part as person ID
            image_id = parts[1].split(".")[0]  # Extract image ID before file extension
            new_name = f"T_D2_{person_id}_{image_id}.jpg"
            shutil.copy(os.path.join(d2_source_real, file_name), os.path.join(target_genuine, new_name))

print("Completed Dataset 2 Process")

Dataset 03: Archive 3

In [ ]:
d3_source_path = os.path.join(resourseDir_path, "archive3/Signature Images")

for file_name in os.listdir(d3_source_path):
    if len(file_name) > 3 and (file_name[2] == "F" or file_name[2] == "T"):  # Ensure filename is long enough to check the third letter and valid format
        file_path = os.path.join(d3_source_path, file_name)

        # identifier = file_name[:3]  # Extract the first three characters for ID
        # count = file_name[4:]  # Extract remaining digits as count
        split_char = "F" if "F" in file_name else "T"
        person_id, image_id_ext = file_name.split(split_char)
        image_id, extension = os.path.splitext(image_id_ext) 

        if file_name[2] == "F":  # Forged signature
            new_name = f"F_D3_{int(person_id) + 30}_{image_id}{extension}"
            shutil.copy(file_path, os.path.join(target_forged, new_name))
        elif file_name[2] == "T":  # Genuine signature
            new_name = f"T_D3_{int(person_id) + 30}_{image_id}{extension}"
            shutil.copy(file_path, os.path.join(target_genuine, new_name))

print("Completed Dataset 3 Process")

Dataset 04: Archive 4

In [ ]:
d4_source_train_path = os.path.join(resourseDir_path, "archive4/sign_data/train")
d4_source_test_path = os.path.join(resourseDir_path, "archive4/sign_data/test")

countRun = 1

def convert_to_jpg(source_path, target_path):
    """Convert an image to .jpg if it's not already"""
    img = Image.open(source_path)
    rgb_img = img.convert("RGB")
    rgb_img.save(target_path, "JPEG")

def d4_organise(main_path, countRun):
    for dirName in os.listdir(main_path):
        dirName_path = os.path.join(main_path, dirName)
        
        if not os.path.isdir(dirName_path):
            print("Non-Identified Resource (Not a directory) --> ", dirName_path)  
            continue  

        dirNameParts = dirName.split("_")

        if len(dirNameParts) == 1:
            file_num = 1
            for fileName in os.listdir(dirName_path):
                if fileName.lower().endswith(".jpg") or fileName.lower().endswith(".png"):
                    fileNameParts = os.path.splitext(fileName)[1].lower()

                    new_name = f"T_D4_{dirNameParts[0]}_{file_num}.jpg"

                    if not fileName.lower().endswith(".jpg"):
                        convert_to_jpg(os.path.join(dirName_path, fileName), os.path.join(target_genuine, new_name))
                    else:
                        shutil.copy(os.path.join(dirName_path, fileName), os.path.join(target_genuine, new_name))
                    
                    file_num += 1
                countRun += 1

            
        else:
            file_num = 1
            for fileName in os.listdir(dirName_path):
                if fileName.lower().endswith(".jpg") or fileName.lower().endswith(".png"):
                    fileNameParts = os.path.splitext(fileName)[1].lower()

                    new_name = f"F_D4_{dirNameParts[0]}_{file_num}.jpg"

                    if not fileName.lower().endswith(".jpg"):
                        convert_to_jpg(os.path.join(dirName_path, fileName), os.path.join(target_forged, new_name))
                    else:
                        shutil.copy(os.path.join(dirName_path, fileName), os.path.join(target_forged, new_name))
                    
                    file_num += 1
                countRun += 1
    return countRun


countRun = d4_organise(d4_source_train_path, countRun)
countRun = d4_organise(d4_source_test_path, countRun)

print("Completed Dataset 4 Process")
print(f"Processed Images Count: {countRun}")


------
Diverse Dataset Creation
---
------

Organising Datasets
---

Moving images from dataset to organised way of dataset (all datasets divided to real and forgery and renamed accordingly)

In [ ]:
# Target Paths
resourseDir_path = "../Dataset/Resources/ProcessingDataset/"

processingDir_path = "../Dataset/Processing"
mainDiverseDir_path = "../Dataset/Processing/DiverseDataset"
organisedDir_path = "../Dataset/Processing/DiverseDataset/OrganisedDatasets"

hybridDir_path = "../Dataset/Processing/DiverseDataset/HybridDatasets"
hybridDir_genuine_path = "../Dataset/Processing/DiverseDataset/HybridDatasets/Genuine"
hybridDir_forged_path = "../Dataset/Processing/DiverseDataset/HybridDatasets/Forged"



# Create target folders if not exist
os.makedirs(mainDiverseDir_path, exist_ok=True)
os.makedirs(organisedDir_path, exist_ok=True)
os.makedirs(hybridDir_path, exist_ok=True)
os.makedirs(hybridDir_genuine_path, exist_ok=True)
os.makedirs(hybridDir_forged_path, exist_ok=True)

print("Completed Dataset Directory Setup")

Dataset 0 (Archive)

In [ ]:
dataset0_path = os.path.join(organisedDir_path, "Dataset0")

d0_source_test_path = os.path.join(resourseDir_path, "archive/Test")
d0_source_train_path = os.path.join(resourseDir_path, "archive/Train")

dataset0_forge_path = os.path.join(dataset0_path, "Forged")
dataset0_genuine_path = os.path.join(dataset0_path, "Genuine")

os.makedirs(dataset0_forge_path, exist_ok=True)
os.makedirs(dataset0_genuine_path, exist_ok=True)

def convert_to_jpg(source_path, target_path):
    """Convert an image to .jpg if it's not already"""
    img = Image.open(source_path)
    rgb_img = img.convert("RGB")  # Ensure it's in RGB mode for .jpg
    rgb_img.save(target_path, "JPEG")

def d0_organise(mainPath, file_cnt):
    for dirName in os.listdir(mainPath):
        dirName_path = os.path.join(mainPath, dirName)

        # Check if it's a directory
        if not os.path.isdir(dirName_path):
            print("Error with unidentified resource -", dirName_path)  
            continue  

        for fileName in os.listdir(dirName_path):
            source_file_path = os.path.join(dirName_path, fileName)
            file_base, file_ext = os.path.splitext(fileName)  # Get base name and extension
            file_ext = file_ext.lower()  # Normalize extension case
            new_name = ""

            if "_" in fileName:
                fileNameParts = file_base.split("_")  # Use file_base to avoid extensions
                if len(fileNameParts) > 2 and fileNameParts[0].lower().startswith("ori"):
                    new_name = f"T_D0_{dirName}_{fileNameParts[2]}"

                elif len(fileNameParts) > 2 and fileNameParts[0].lower().startswith("for"):
                    new_name = f"F_D0_{dirName}_{fileNameParts[2]}"

            elif "-" in fileName:
                fileNameParts = file_base.split("-")
                if len(fileNameParts) >= 5:
                    if fileNameParts[3].lower().startswith("g"):
                        new_name = f"T_D0_{fileNameParts[2]}_{fileNameParts[4]}"

                    elif fileNameParts[3].lower().startswith("f"):
                        new_name = f"F_D0_{fileNameParts[2]}_{fileNameParts[4]}"

            if new_name:
                new_name += ".jpg"  # Ensure final filename ends with .jpg
                target_dir = dataset0_genuine_path if new_name.startswith("T_") else dataset0_forge_path
                target_file_path = os.path.join(target_dir, new_name)

                if file_ext != ".jpg":
                    convert_to_jpg(source_file_path, target_file_path)
                else:
                    shutil.copy(source_file_path, target_file_path)

                file_cnt += 1

    return file_cnt

d0_file_cnt = 0
d0_file_cnt = d0_organise(d0_source_test_path, d0_file_cnt)
d0_file_cnt = d0_organise(d0_source_train_path, d0_file_cnt)

print("Completed Dataset 2 Process")
print(f"Images Count: {d0_file_cnt}")


Dataset 2 (Archive 2)

In [ ]:
dataset2_path = os.path.join(organisedDir_path, "Dataset2")

dataset2_forge_path = os.path.join(dataset2_path, "Forged")
dataset2_genuine_path = os.path.join(dataset2_path, "Genuine")

d2_source_forgery = os.path.join(resourseDir_path, "archive2/FORGERY_ALL")
d2_source_real = os.path.join(resourseDir_path, "archive2/REAL_ALL")

os.makedirs(dataset2_forge_path, exist_ok=True)
os.makedirs(dataset2_genuine_path, exist_ok=True)

# Processing Forged Signatures
for file_name in os.listdir(d2_source_forgery):
    if file_name.endswith(".jpg"):
        parts = file_name.split("_")
        if len(parts) == 2:
            person_id = parts[0][-3:]  # Extract last 3 digits of the first part as person ID
            image_id = parts[1].split(".")[0]  # Extract image ID before file extension
            new_name = f"F_D2_{person_id}_{image_id}.jpg"
            shutil.copy(os.path.join(d2_source_forgery, file_name), os.path.join(dataset2_forge_path, new_name))

# Processing Genuine Signatures
for file_name in os.listdir(d2_source_real):
    if file_name.endswith(".jpg"):
        parts = file_name.split("_")
        if len(parts) == 2:
            person_id = parts[0][-3:]  # Extract last 3 digits of the first part as person ID
            image_id = parts[1].split(".")[0]  # Extract image ID before file extension
            new_name = f"T_D2_{person_id}_{image_id}.jpg"
            shutil.copy(os.path.join(d2_source_real, file_name), os.path.join(dataset2_genuine_path, new_name))

print("Completed Dataset 2 Process")

Dataset 3 (Archive 3)

In [ ]:
dataset3_path = os.path.join(organisedDir_path, "Dataset3")

dataset3_forge_path = os.path.join(dataset3_path, "Forged")
dataset3_genuine_path = os.path.join(dataset3_path, "Genuine")

d3_source_path = os.path.join(resourseDir_path, "archive3/Signature Images")

os.makedirs(dataset3_forge_path, exist_ok=True)
os.makedirs(dataset3_genuine_path, exist_ok=True)

for file_name in os.listdir(d3_source_path):
    if len(file_name) > 3 and (file_name[2] == "F" or file_name[2] == "T"):  # Ensure filename is long enough to check the third letter and valid format
        file_path = os.path.join(d3_source_path, file_name)

        # identifier = file_name[:3]  # Extract the first three characters for ID
        # count = file_name[4:]  # Extract remaining digits as count
        split_char = "F" if "F" in file_name else "T"
        person_id, image_id_ext = file_name.split(split_char)
        image_id, extension = os.path.splitext(image_id_ext) 

        if file_name[2] == "F":  # Forged signature
            new_name = f"F_D3_{int(person_id) + 30}_{image_id}{extension}"
            shutil.copy(file_path, os.path.join(dataset3_forge_path, new_name))
        elif file_name[2] == "T":  # Genuine signature
            new_name = f"T_D3_{int(person_id) + 30}_{image_id}{extension}"
            shutil.copy(file_path, os.path.join(dataset3_genuine_path, new_name))

print("Completed Dataset 3 Process")

Dataset 4 (Archive 4)

In [ ]:
dataset4_path = os.path.join(organisedDir_path, "Dataset4")

dataset4_forge_path = os.path.join(dataset4_path, "Forged")
dataset4_genuine_path = os.path.join(dataset4_path, "Genuine")

d4_source_train_path = os.path.join(resourseDir_path, "archive4/sign_data/train")
d4_source_test_path = os.path.join(resourseDir_path, "archive4/sign_data/test")

os.makedirs(dataset4_forge_path, exist_ok=True)
os.makedirs(dataset4_genuine_path, exist_ok=True)

countRun = 1

def convert_to_jpg(source_path, target_path):
    """Convert an image to .jpg if it's not already"""
    img = Image.open(source_path)
    rgb_img = img.convert("RGB")
    rgb_img.save(target_path, "JPEG")

def d4_organise(main_path, countRun):
    for dirName in os.listdir(main_path):
        dirName_path = os.path.join(main_path, dirName)
        
        if not os.path.isdir(dirName_path):
            print("Non-Identified Resource (Not a directory) --> ", dirName_path)  
            continue  

        dirNameParts = dirName.split("_")

        if len(dirNameParts) == 1:
            file_num = 1
            for fileName in os.listdir(dirName_path):
                if fileName.lower().endswith(".jpg") or fileName.lower().endswith(".png"):
                    fileNameParts = os.path.splitext(fileName)[1].lower()

                    new_name = f"T_D4_{dirNameParts[0]}_{file_num}.jpg"

                    if not fileName.lower().endswith(".jpg"):
                        convert_to_jpg(os.path.join(dirName_path, fileName), os.path.join(dataset4_genuine_path, new_name))
                    else:
                        shutil.copy(os.path.join(dirName_path, fileName), os.path.join(dataset4_genuine_path, new_name))
                    
                    file_num += 1
                countRun += 1

            
        else:
            file_num = 1
            for fileName in os.listdir(dirName_path):
                if fileName.lower().endswith(".jpg") or fileName.lower().endswith(".png"):
                    fileNameParts = os.path.splitext(fileName)[1].lower()

                    new_name = f"F_D4_{dirNameParts[0]}_{file_num}.jpg"

                    if not fileName.lower().endswith(".jpg"):
                        convert_to_jpg(os.path.join(dirName_path, fileName), os.path.join(dataset4_forge_path, new_name))
                    else:
                        shutil.copy(os.path.join(dirName_path, fileName), os.path.join(dataset4_forge_path, new_name))
                    
                    file_num += 1
                countRun += 1
    return countRun


countRun = d4_organise(d4_source_train_path, countRun)
countRun = d4_organise(d4_source_test_path, countRun)

print("Completed Dataset 4 Process")
print(f"Processed Images Count: {countRun}")


Creating New Hybrid Dataset

In [ ]:
def createHybridDatast(path, isGenuine):
    dir_files = os.listdir(path)
    dir_file_cnt = len(dir_files)

    print(f"Files count: {dir_file_cnt}")

    # Use a set for faster duplicate checking
    list_random = set()

    while len(list_random) < 400 and len(list_random) < dir_file_cnt:
        random_num = random.randint(0, dir_file_cnt - 1)
        list_random.add(random_num)

    random_filenames = [dir_files[i] for i in list_random]

    # Copy selected files to the new directory
    for file in random_filenames:
        shutil.copy(
            os.path.join(path, file),
            os.path.join(hybridDir_genuine_path if isGenuine else hybridDir_forged_path, file)
        )

    print(random_filenames)

path_list = [
    [dataset0_genuine_path, True],
    [dataset0_forge_path, False],
    [dataset2_genuine_path, True],
    [dataset2_forge_path, False],
    [dataset3_genuine_path, True],
    [dataset3_forge_path, False],
    [dataset4_genuine_path, True],
    [dataset4_forge_path, False],
]

for path in path_list:
    createHybridDatast(path[0], path[1])